# Field deploy — ship a validated field as a queryable, calibrated column

Run **after** `field_factory.ipynb` and after the human label review. The quality gate is
explicit: this notebook refuses to run without a reviewed labels file, and prints the
validation numbers (incl. flip rate) before doing anything expensive.

Steps:
1. **Gate** — load reviewed labels, show flip rate + F1; deployment calibration = isotonic
   refit on *all* reviewed labels (validation already happened out-of-fold in the factory).
2. **Full-corpus pass** — LLM-extract every candidate case (checkpointed; the 120 sampled
   cases are already in the shared checkpoint and are not re-run).
3. **Deployed column** — `data/field_<name>_deployed.parquet`: label, raw + calibrated
   confidence, evidence, provenance per case; non-candidates are False/0.02.
4. **Sidecar meta** — `data/field_<name>_meta.json`: definition, lexicon, threshold,
   validation numbers. **`ask.ipynb` discovers deployed fields through these sidecars** and
   routes matching questions to the calibrated handler automatically — define → review →
   deploy → query, no further wiring.
5. **Demo** — the threshold-aware count with abstention audit, straight from the new column.

## 1. Config + gate (reuses the factory's field definition by exec)

In [ ]:
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd

FIELD_NAME_DEPLOY = "supervised_contact"      # must match the factory run

# exec the factory's config/lexicon/detector cells — single source of truth
src_f = json.loads(Path("field_factory.ipynb").read_text())
_g = {"re": re, "json": json}
for marker in ['FIELD_NAME = "', "LEXICON = list(SEED_TERMS)", "def detect(full_text)"]:
    cell = next("".join(c["source"]) for c in src_f["cells"]
                if c["cell_type"] == "code" and marker in "".join(c["source"]))
    exec(cell, _g)
assert _g["FIELD_NAME"] == FIELD_NAME_DEPLOY, \
    f"factory config is for {_g['FIELD_NAME']!r} — update one of the two"
FIELD_DEFINITION, LEXICON, LEX_RE = _g["FIELD_DEFINITION"], _g["LEXICON"], _g["LEX_RE"]
detect, det = _g["detect"], _g["det"]
LABELS   = Path(f"../data/field_{FIELD_NAME_DEPLOY}_labels.csv")
CKPT     = Path(f"../data/field_{FIELD_NAME_DEPLOY}_llm_checkpoint.json")
OUT_PARQ = Path(f"../data/field_{FIELD_NAME_DEPLOY}_deployed.parquet")
OUT_META = Path(f"../data/field_{FIELD_NAME_DEPLOY}_meta.json")
THRESHOLD = 0.7
LLM_MODEL, MAX_SENTS = _g["LLM_MODEL"], _g["MAX_SENTS"]

# ---- the quality gate ----
if not LABELS.exists():
    raise SystemExit(f"GATE: no reviewed labels ({LABELS.name}). Run the factory, review "
                     f"the template, save the labels — then deploy.")
lab = pd.read_csv(LABELS)
gcol = f"gold_{FIELD_NAME_DEPLOY}"
lab["gold"] = pd.to_numeric(lab[gcol], errors="coerce")
lab = lab[lab.gold.notna()].copy()
lab["gold"] = lab.gold.astype(int)
flips = int((lab.gold != lab.draft_label).sum())
from sklearn.metrics import precision_recall_fscore_support
p, r, f1, _ = precision_recall_fscore_support(lab.gold, lab.draft_label.astype(int),
                                              average="binary", zero_division=0)
print(f"GATE OK: {len(lab)} reviewed labels | flip rate {flips}/{len(lab)} "
      f"({flips/len(lab)*100:.0f}%) | LLM F1 vs gold {f1:.3f} (P {p:.3f} / R {r:.3f})")
if flips == 0:
    print("!! WARNING: zero flips — labels may be rubber-stamped; validation is weak.")

# deployment calibration: refit on ALL reviewed labels (validated OOF in the factory)
from sklearn.isotonic import IsotonicRegression
iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
iso.fit(lab.draft_conf.to_numpy(float), lab.gold.to_numpy(float))
print("deployment isotonic fitted on all reviewed labels")

## 2. Full-corpus pass (checkpointed; sampled cases already cached)

In [ ]:
try:
    import ollama
    ollama.list()
    OLLAMA_OK = True
except Exception as e:
    raise SystemExit(f"Ollama unreachable ({e}) — start `ollama serve` for the full pass")

PROMPT = ("You classify excerpts from an ECHR family-law case.\n\nDefinition: "
          + FIELD_DEFINITION +
          "\n\nSentences from the case (with document section):\n{sents}\n\n"
          "Reply ONLY with a JSON object: "
          '{{"label": true or false, "confidence": 0.0-1.0, '
          '"evidence": "<best supporting sentence, verbatim>"}}')
_JSON = re.compile(r"\{.*\}", re.S)

done = json.loads(CKPT.read_text()) if CKPT.exists() else {}
candidates = [iid for iid, d in det.items() if d["pool"]]
todo = [i for i in candidates if i not in done]
print(f"candidates: {len(candidates)} | cached: {len(done)} | to run: {len(todo)}")
import time
t0 = time.time()
for n, iid in enumerate(todo, 1):
    sents = "\n".join(f"- [{lab_}] {s}" for lab_, s in det[iid]["pool"])
    out = None
    for _ in range(2):
        try:
            rr = ollama.chat(model=LLM_MODEL,
                             messages=[{"role": "user", "content": PROMPT.format(sents=sents)}],
                             options={"temperature": 0})
            m = _JSON.search(rr["message"]["content"])
            obj = json.loads(m.group(0))
            out = {"label": bool(obj.get("label")),
                   "conf": max(0.0, min(1.0, float(obj.get("confidence", 0.5)))),
                   "evidence": str(obj.get("evidence", ""))[:300]}
            break
        except Exception:
            pass
    done[iid] = out or {"label": False, "conf": 0.5, "evidence": "unparseable"}
    if n % 10 == 0 or n == len(todo):
        CKPT.write_text(json.dumps(done))
        rate = (time.time() - t0) / n
        print(f"  {n}/{len(todo)}  ({rate:.1f}s/case, ~{rate*(len(todo)-n)/60:.0f} min left)")
print("full pass complete")

## 3. Deployed column + sidecar meta

In [ ]:
rows = []
for iid, d in det.items():
    if d["pool"] and iid in done:
        o = done[iid]
        raw = float(o["conf"])
        rows.append({"id": iid, FIELD_NAME_DEPLOY: bool(o["label"]),
                     "conf_raw": raw, "conf_cal": float(iso.predict([raw])[0]),
                     "evidence": o.get("evidence"),
                     "provenance": f"llm:{LLM_MODEL};sents={len(d['pool'])}"})
    else:
        rows.append({"id": iid, FIELD_NAME_DEPLOY: False, "conf_raw": 0.02,
                     "conf_cal": float(iso.predict([0.02])[0]), "evidence": None,
                     "provenance": "no-lexicon-mention"})
dep = pd.DataFrame(rows)
dep.to_parquet(OUT_PARQ, index=False)
print(f"wrote {OUT_PARQ.name}: {len(dep)} rows | positive: {int(dep[FIELD_NAME_DEPLOY].sum())} "
      f"| confident (cal>={THRESHOLD}): {int((dep[FIELD_NAME_DEPLOY] & (dep.conf_cal >= THRESHOLD)).sum())}")

meta = {"field": FIELD_NAME_DEPLOY, "definition": FIELD_DEFINITION,
        "lexicon": sorted(set(LEXICON)), "threshold": THRESHOLD,
        "corpus": "echr_parental_alienation.json (ECHR-EN; domain of validity)",
        "validation": {"n_labels": int(len(lab)), "flip_rate": f"{flips}/{len(lab)}",
                       "llm_f1": round(float(f1), 3)},
        "labels_file": LABELS.name, "model": LLM_MODEL}
OUT_META.write_text(json.dumps(meta, indent=2))
print(f"wrote {OUT_META.name} — ask.ipynb discovers the field through this sidecar")

## 4. Demo — the new field answering its Bucket-3 question, threshold-aware

In [ ]:
import duckdb
con = duckdb.connect()
con.register("f", dep)
q = f"""SELECT COUNT(*) FILTER ({FIELD_NAME_DEPLOY} AND conf_cal >= {THRESHOLD}) AS confident,
               COUNT(*) FILTER ({FIELD_NAME_DEPLOY} AND conf_cal <  {THRESHOLD}) AS abstained,
               COUNT(*) AS corpus
        FROM f"""
res = con.execute(q).df()
print(f'Q: "How many cases involve {FIELD_NAME_DEPLOY.replace("_", " ")}?"')
print(res.to_string(index=False))
print(f"\nANSWER pattern: <confident> cases at calibrated conf >= {THRESHOLD}; "
      f"<abstained> predicted-positive cells abstained (surfaced, not dropped).")
print(f"CAVEAT: LLM extractor, F1 {f1:.2f} on {len(lab)} reviewed labels "
      f"(flip rate {flips}/{len(lab)}); ECHR-English only.")